# Paso 1: Definición del problema:

En esta sección se identifica una necesidad específica y se transforma en un problema de Machine Learning. El conjunto de datos seleccionado debe contar, preferiblemente, con al menos 60,000 instancias y 20 variables predictoras (incluyendo una categórica) para asegurar la robustez del modelo.

-------------------------------------------------

# Paso 2: Obtencion y carga del conjunto de datos:
- El conjunto de datos se obtuvo desde la página de Kaggle, se encuentra almacenado en un formato ".csv".
"https://www.kaggle.com/datasets/zongaobian/h1b-lca-disclosure-data-2020-2024?select=Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv"



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### A. Carga del conjunto de datos

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import xgboost as xgb
import optuna
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV


file_path = '/content/drive/MyDrive/Colab Notebooks/PROYECTO 4GEEKS/Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv'

In [3]:
use_cols = [
    # Target y unidad del target
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",

    # Salario de referencia regulatorio
    "PREVAILING_WAGE",
    "PW_UNIT_OF_PAY",
    "PW_WAGE_LEVEL",
    "PW_OES_YEAR",
    "PW_OTHER_SOURCE",

    # Fechas útiles para crear variables temporales
    "RECEIVED_DATE",
    "BEGIN_DATE",
    "END_DATE",

    # Ubicación del empleador y del trabajo
    "EMPLOYER_STATE",
    "WORKSITE_STATE",
    "WORKSITE_CITY",
    "WORKSITE_COUNTY",

    # Visa y ocupación
    "VISA_CLASS",
    "JOB_TITLE",
    "SOC_CODE",
    "SOC_TITLE",
    "NAICS_CODE",

    # Condiciones del empleo
    "FULL_TIME_POSITION",
    "TOTAL_WORKER_POSITIONS",
    "NEW_EMPLOYMENT",
    "CONTINUED_EMPLOYMENT",
    "CHANGE_PREVIOUS_EMPLOYMENT",
    "NEW_CONCURRENT_EMPLOYMENT",
    "CHANGE_EMPLOYER",
    "AMENDED_PETITION",

    # Lugar de trabajo y estructura
    "TOTAL_WORKSITE_LOCATIONS",
    "WORKSITE_WORKERS",
    "SECONDARY_ENTITY",

    # Indicadores regulatorios del empleador
    "H_1B_DEPENDENT",
    "WILLFUL_VIOLATOR",
    "SUPPORT_H1B",
    "STATUTORY_BASIS",

    # Útil para EDA, no necesariamente para modelo base
    "EMPLOYER_NAME",

    # Estado del caso, útil para filtrar o analizar
    "CASE_STATUS"
]

In [4]:
df_reduced = pd.read_csv(
    file_path,
    usecols=use_cols,
    low_memory=False
)
df_reduced.shape

(3564698, 36)

In [5]:
df_reduced.head()

,CASE_STATUS,RECEIVED_DATE,VISA_CLASS,JOB_TITLE,SOC_CODE,SOC_TITLE,FULL_TIME_POSITION,BEGIN_DATE,END_DATE,TOTAL_WORKER_POSITIONS,...,PREVAILING_WAGE,PW_UNIT_OF_PAY,PW_WAGE_LEVEL,PW_OES_YEAR,PW_OTHER_SOURCE,TOTAL_WORKSITE_LOCATIONS,H_1B_DEPENDENT,WILLFUL_VIOLATOR,SUPPORT_H1B,STATUTORY_BASIS
0,Certified,2019-09-25,H-1B,"APPLICATION ENGINEER, OMS [15-1199.02]",15-1199,"COMPUTER OCCUPATIONS, ALL OTHER",Y,2019-10-07,2022-10-07,1,...,95118.0,Year,IV,2018.0,OES,NaN,N,N,NaN,NaN
1,Certified,2019-09-25,H-1B,BI DEVELOPER II,15-1132,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,2020-01-08,2023-01-07,1,...,39.0,Hour,II,2019.0,OES,NaN,Y,N,Y,BOTH
2,Certified,2019-09-25,H-1B,QUALITY ENGINEER,17-2141,MECHANICAL ENGINEERS,Y,2019-10-03,2022-10-02,1,...,39.0,Hour,II,2019.0,OES,NaN,Y,N,Y,BOTH
3,Certified,2019-09-25,H-1B,"SOFTWARE DEVELOPER, APPLICATIONS",15-1132,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,2019-10-07,2022-10-01,1,...,53.0,Hour,IV,2019.0,OES,NaN,Y,N,Y,BOTH
4,Certified,2019-09-25,H-1B,QUALITY ENGINEER LEVEL II,15-1199,"COMPUTER OCCUPATIONS, ALL OTHER",Y,2019-10-09,2022-10-08,1,...,65333.0,Year,II,2019.0,OES,NaN,Y,N,Y,BOTH


## Paso 3: Almacenar la información

Para garantizar la seguridad y eficiencia en el acceso, los datos se migran del formato plano (CSV) a una base de datos relacional o no relacional. Se realizan consultas (SELECT, JOIN, INSERT) mediante Python para validar la estructura y preparar la información antes del análisis estadístico.

--------------------------------

## Paso 4: Realiza un análisis descriptivo

Se exploran las medidas estadísticas fundamentales de las variables predictoras, tales como medias, distribuciones y desviaciones típicas. El objetivo es comprender la naturaleza de los datos y teorizar sobre sus distribuciones de probabilidad, utilizando contrastes de hipótesis si es necesario.

In [6]:
df_reduced.describe().T

,count,mean,std,min,25%,50%,75%,max
TOTAL_WORKER_POSITIONS,3564698.0,1.762165,5.898817,1.00,1.0,1.0,1.0,1.098000e+03
NEW_EMPLOYMENT,3564698.0,0.635176,3.915777,0.00,0.0,0.0,1.0,1.098000e+03
CONTINUED_EMPLOYMENT,3564698.0,0.371526,2.468180,0.00,0.0,0.0,1.0,1.098000e+03
CHANGE_PREVIOUS_EMPLOYMENT,3564698.0,0.148591,1.311127,0.00,0.0,0.0,0.0,1.098000e+03
NEW_CONCURRENT_EMPLOYMENT,3564698.0,0.010089,0.723727,0.00,0.0,0.0,0.0,1.098000e+03
CHANGE_EMPLOYER,3564698.0,0.304038,1.626940,0.00,0.0,0.0,0.0,1.098000e+03
AMENDED_PETITION,3564698.0,0.299699,1.490564,0.00,0.0,0.0,0.0,1.098000e+03
NAICS_CODE,3564698.0,424819.490078,208648.607037,-271021.00,334413.0,541511.0,541512.0,5.698555e+06
WORKSITE_WORKERS,3551363.0,1.761110,5.891472,1.00,1.0,1.0,1.0,1.098000e+03
WAGE_RATE_OF_PAY_FROM,3564696.0,112164.622733,650897.510530,7.25,80779.0,104749.0,139000.0,1.204781e+09


In [7]:
df_reduced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3564698 entries, 0 to 3564697
Data columns (total 36 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   CASE_STATUS                 object 
 1   RECEIVED_DATE               object 
 2   VISA_CLASS                  object 
 3   JOB_TITLE                   object 
 4   SOC_CODE                    object 
 5   SOC_TITLE                   object 
 6   FULL_TIME_POSITION          object 
 7   BEGIN_DATE                  object 
 8   END_DATE                    object 
 9   TOTAL_WORKER_POSITIONS      int64  
 10  NEW_EMPLOYMENT              int64  
 11  CONTINUED_EMPLOYMENT        int64  
 12  CHANGE_PREVIOUS_EMPLOYMENT  int64  
 13  NEW_CONCURRENT_EMPLOYMENT   int64  
 14  CHANGE_EMPLOYER             int64  
 15  AMENDED_PETITION            int64  
 16  EMPLOYER_NAME               object 
 17  EMPLOYER_STATE              object 
 18  NAICS_CODE                  int64  
 19  WORKSITE_WORKERS     

## Paso 5: Realiza un EDA completo

Análisis Exploratorio de Datos (EDA) para identificar y seleccionar las variables más relevantes, eliminando el ruido o información redundante. En esta fase se realiza también la división crítica del conjunto de datos en subconjuntos de entrenamiento (train) y prueba (test).

### Analizamos columnas con valores nulos

In [8]:
col_nulls = pd.DataFrame(df_reduced.isnull().sum()).sort_values(0,ascending=False)
col_nulls

,0
PW_OTHER_SOURCE,3332574
STATUTORY_BASIS,2571646
SUPPORT_H1B,2565205
PW_WAGE_LEVEL,248298
PW_OES_YEAR,230239
WILLFUL_VIOLATOR,91302
H_1B_DEPENDENT,91280
WORKSITE_WORKERS,13335
TOTAL_WORKSITE_LOCATIONS,13333
PW_UNIT_OF_PAY,1962


### Analizamos columnas con alto % de valores nulos

In [9]:
df_reduced["PW_OTHER_SOURCE"].value_counts()

,count
PW_OTHER_SOURCE,
Survey,185369
CBA,34805
OES,10917
SURVEY,355
SCA,353
DBA,285
PWD,40


In [10]:
df_reduced["STATUTORY_BASIS"].value_counts()


,count
STATUTORY_BASIS,
"$60,000 or higher annual wage",565412
"Both $60,000 or higher in annual wage and Masters Degree or higher in related specialty",204491
WAGE,141356
BOTH,53515
Wage,17410
Both,7226
Masters Degree or higher in related specialty,2475
DEGREE,1093
Degree,74


In [11]:
df_reduced["SUPPORT_H1B"].value_counts()


,count
SUPPORT_H1B,
Yes,772384
Y,221127
No,4555
N,1427


In [12]:
df_reduced["PW_WAGE_LEVEL"].value_counts()

,count
PW_WAGE_LEVEL,
II,1508404
III,734226
I,540676
IV,533073
V,21


In [13]:
df_reduced["PW_OES_YEAR"].value_counts()


,count
PW_OES_YEAR,
7/1/2023 - 6/30/2024,794161
7/1/2020 - 6/30/2021,695132
7/1/2021 - 6/30/2022,624346
7/1/2022 - 6/30/2023,591171
7/1/2019 - 6/30/2020,458999
7/1/2024 - 6/30/2025,94860
10/08/2020 - 6/30/2021,64873
2019.0,4876
2020.0,2502


### Normalizando la columna "PW_OES_YEAR"

In [14]:
import re
def extraer_pw_oes_period(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    years = re.findall(r"\b(20\d{2})\b", value)

    if len(years) >= 2:
        return f"{years[0]}-{years[1]}"

    if len(years) == 1:
        start_year = int(years[0])
        return f"{start_year}-{start_year + 1}"

    return np.nan


df_reduced["pw_oes_period_group"] = df_reduced["PW_OES_YEAR"].apply(extraer_pw_oes_period)

In [15]:
valid_periods = [
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
    "2024-2025"
]

df_reduced["pw_oes_period_group"] = df_reduced["pw_oes_period_group"].where(
    df_reduced["pw_oes_period_group"].isin(valid_periods) | df_reduced["pw_oes_period_group"].isna(),
    np.nan
)

In [16]:
df_reduced["pw_oes_period_group"].value_counts(dropna=False).sort_index()

,count
pw_oes_period_group,
2018-2019,1106
2019-2020,465235
2020-2021,763579
2021-2022,624346
2022-2023,591171
2023-2024,794161
2024-2025,94860
NaN,230240


#### La variable `PW_OES_YEAR` fue transformada en `pw_oes_period_group`, conservando el periodo salarial de referencia en formato `YYYY-YYYY`. Esta decisión facilita la interpretación del análisis, ya que la variable original representa periodos de vigencia salarial y no años calendario aislados.

In [17]:
cols_to_drop = [
    "PW_OTHER_SOURCE",
    "STATUTORY_BASIS",
    "SUPPORT_H1B",
    "PW_OES_YEAR"
]

#### Se han eliminado inicialmente las 4 columnas con mayor cantidad de datos nulos y que con previo análisis confirmamos que no entregan información relevante para el modelo.

In [18]:
df_reduced = df_reduced.drop(columns=cols_to_drop, errors="ignore")

In [19]:
df_reduced.shape

(3564698, 33)

---------------------------------------
---------------------------------------

### Analisis y limpieza de la columna "SOC_TITLE"

In [20]:
df_reduced["SOC_TITLE"].value_counts()


,count
SOC_TITLE,
"Software Developers, Applications",633356
Software Developers,527845
Computer Systems Analysts,194637
Computer Systems Engineers/Architects,147930
"Software Developers, Systems Software",120693
...,...
Social And Community Service Managers,1
Water/Wastewater Engineer,1
Validation Engineers,1


---------------------------
---------------------------


## Analisis y Limpieza de la columna "SOC_CODE"

In [21]:
df_reduced["SOC_CODE"].value_counts()

,count
SOC_CODE,
15-1132.00,597833
15-1252.00,525445
15-1121.00,120539
15-1133.00,114309
15-1299.08,85680
...,...
49-9061.00,1
43-4031.01,1
31-2011.00,1


In [22]:
top_soc_combinations = (
    df_reduced
    .groupby(["SOC_CODE", "SOC_TITLE"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

display(top_soc_combinations)

,SOC_CODE,SOC_TITLE,count
662,15-1132.00,"Software Developers, Applications",597648
989,15-1252.00,Software Developers,525286
556,15-1121.00,Computer Systems Analysts,120383
694,15-1133.00,"Software Developers, Systems Software",114253
1089,15-1299.08,Computer Systems Engineers/Architects,85616
60,11-3021.00,Computer and Information Systems Managers,77186
895,15-1211.00,Computer Systems Analysts,64902
837,15-1199.02,Computer Systems Engineers/Architects,59637
1382,17-2141.00,Mechanical Engineers,58961
1131,15-2031.00,Operations Research Analysts,57711


## Se normalizarán los datos de SOC_CODE ya que hay códigos repetidos en diferentes formatos y código contaminado con texto.

In [23]:
# Limpiar SOC_CODE
df_reduced["soc_code_clean"] = (
    df_reduced["SOC_CODE"]
    .astype("string")
    .str.strip()
    .str.extract(r"(\d{2}-\d{4}(?:\.\d{2})?)")[0]
)
# Eliminar SOC_CODE original
df_reduced = df_reduced.drop(columns=["SOC_CODE"], errors="ignore")

In [24]:
print("Categorías en soc_code_clean:")
print(df_reduced["soc_code_clean"].nunique())

Categorías en soc_code_clean:
1420


In [25]:
# Frecuencia de cada código SOC limpio
soc_code_counts = df_reduced["soc_code_clean"].value_counts(dropna=False)

# Distribución de frecuencias
soc_code_freq_distribution = (
    soc_code_counts
    .value_counts()
    .sort_index()
    .rename_axis("cantidad_de_apariciones")
    .reset_index(name="cantidad_de_codigos")
)

display(soc_code_freq_distribution.head(10))

,cantidad_de_apariciones,cantidad_de_codigos
0,1,147
1,2,74
2,3,59
3,4,43
4,5,32
5,6,26
6,7,28
7,8,21
8,9,19
9,10,27


### Distribución de valores de "soc_code_clean"

In [26]:
soc_code_freq_df = soc_code_counts.reset_index()
soc_code_freq_df.columns = ["soc_code_clean", "count"]

fig = px.histogram(
    soc_code_freq_df,
    x="count",
    nbins=100,
    log_y=True,
    title="Distribución de frecuencias de soc_code_clean",
    labels={
        "count": "Cantidad de registros por código SOC",
        "y": "Cantidad de códigos SOC"
    }
)

fig.update_layout(
    xaxis_title="Cantidad de veces que aparece un código SOC",
    yaxis_title="Cantidad de códigos SOC, escala log",
    height=500
)

fig.show()

In [27]:
thresholds = [100, 250, 500, 1000, 1500, 5000, 10000]

rare_summary_soc_code = []

for t in thresholds:
    rare_categories = (soc_code_counts < t).sum()
    rare_rows = soc_code_counts[soc_code_counts < t].sum()

    rare_summary_soc_code.append({
        "umbral_menor_a": t,
        "codigos_agrupados": rare_categories,
        "porcentaje_codigos": round(rare_categories / len(soc_code_counts) * 100, 2),
        "filas_agrupadas": rare_rows,
        "porcentaje_filas": round(rare_rows / len(df_reduced) * 100, 2)
    })

rare_summary_soc_code_df = pd.DataFrame(rare_summary_soc_code)

display(rare_summary_soc_code_df)

,umbral_menor_a,codigos_agrupados,porcentaje_codigos,filas_agrupadas,porcentaje_filas
0,100,898,63.19,18129,0.51
1,250,1024,72.06,38282,1.07
2,500,1126,79.24,75202,2.11
3,1000,1197,84.24,127097,3.57
4,1500,1243,87.47,184174,5.17
5,5000,1332,93.74,420022,11.78
6,10000,1368,96.27,679422,19.06


In [28]:
min_count = 1500

valid_soc_codes = df_reduced["soc_code_clean"].value_counts()
valid_soc_codes = valid_soc_codes[valid_soc_codes >= min_count].index

df_reduced["soc_code_grouped"] = df_reduced["soc_code_clean"].where(
    df_reduced["soc_code_clean"].isin(valid_soc_codes),
    "OTHER"
)

df_reduced = df_reduced.drop(columns=["SOC_CODE", "soc_code_clean"], errors="ignore")


In [29]:
print("Categorías finales en soc_code_grouped:")
print(df_reduced["soc_code_grouped"].nunique())

Categorías finales en soc_code_grouped:
179


In [30]:
print("Filas agrupadas como OTHER:")
print((df_reduced["soc_code_grouped"] == "OTHER").sum())

Filas agrupadas como OTHER:
184174


In [31]:
df_reduced["soc_code_grouped"].value_counts()

,count
soc_code_grouped,
15-1132.00,599606
15-1252.00,525445
OTHER,184174
15-1121.00,120539
15-1133.00,114438
...,...
13-2011,1536
15-1141,1523
29-1062.00,1517


In [32]:
soc_code_grouped_counts = df_reduced["soc_code_grouped"].value_counts(dropna=False)
soc_code_grouped_freq_df = soc_code_grouped_counts.reset_index()
soc_code_grouped_freq_df.columns = ["soc_code_grouped", "count"]

fig = px.histogram(
    soc_code_grouped_freq_df,
    x="count",
    nbins=100,
    log_y=True,
    title="Distribución de frecuencias de soc_code_grouped",
    labels={
        "count": "Cantidad de registros por código SOC",
        "y": "Cantidad de códigos SOC agrupados"
    }
)

fig.update_layout(
    xaxis_title="Cantidad de veces que aparece un código SOC",
    yaxis_title="Cantidad de códigos SOC agrupados",
    height=500
)

fig.update_yaxes(
    tickmode="array",
    tickvals=[1, 10, 100, 1000],
    ticktext=["1", "10", "100", "1.000"]
)

fig.show()

La variable `SOC_CODE` presentaba alta cardinalidad y formatos inconsistentes, como códigos combinados con texto, generando `soc_code_clean`.

Luego se analizó la frecuencia de cada código limpio. Se observó que muchos códigos aparecían muy pocas veces, por lo que se creó `soc_code_grouped`, manteniendo solo códigos con al menos 500 registros y agrupando el resto como `OTHER`.

Con este umbral se agrupa el 79.24% de los códigos de categorías de trabajos, pero solo el 2.11% de las filas totales, reduciendo dimensionalidad sin afectar significativamente la información del dataset.

------------------------------
------------------------------

## Creación de variable objetivo a raiz de la columna WAGE_RATE_OF_PAY_FROM y WAGE_UNIT_OF_PAY

Haremos una conversión de valores donde normalizaremos los valores desde días, semanas, bi-semanas y horas, a que todos los valores queden anuales.

In [33]:
df_reduced.head()

,CASE_STATUS,RECEIVED_DATE,VISA_CLASS,JOB_TITLE,SOC_TITLE,FULL_TIME_POSITION,BEGIN_DATE,END_DATE,TOTAL_WORKER_POSITIONS,NEW_EMPLOYMENT,...,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY,PREVAILING_WAGE,PW_UNIT_OF_PAY,PW_WAGE_LEVEL,TOTAL_WORKSITE_LOCATIONS,H_1B_DEPENDENT,WILLFUL_VIOLATOR,pw_oes_period_group,soc_code_grouped
0,Certified,2019-09-25,H-1B,"APPLICATION ENGINEER, OMS [15-1199.02]","COMPUTER OCCUPATIONS, ALL OTHER",Y,2019-10-07,2022-10-07,1,1,...,100000.00,Year,95118.0,Year,IV,NaN,N,N,2018-2019,15-1199
1,Certified,2019-09-25,H-1B,BI DEVELOPER II,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,2020-01-08,2023-01-07,1,0,...,38.57,Hour,39.0,Hour,II,NaN,Y,N,2019-2020,15-1132
2,Certified,2019-09-25,H-1B,QUALITY ENGINEER,MECHANICAL ENGINEERS,Y,2019-10-03,2022-10-02,1,0,...,43.50,Hour,39.0,Hour,II,NaN,Y,N,2019-2020,17-2141
3,Certified,2019-09-25,H-1B,"SOFTWARE DEVELOPER, APPLICATIONS","SOFTWARE DEVELOPERS, APPLICATIONS",Y,2019-10-07,2022-10-01,1,0,...,57.69,Hour,53.0,Hour,IV,NaN,Y,N,2019-2020,15-1132
4,Certified,2019-09-25,H-1B,QUALITY ENGINEER LEVEL II,"COMPUTER OCCUPATIONS, ALL OTHER",Y,2019-10-09,2022-10-08,1,0,...,75000.00,Year,65333.0,Year,II,NaN,Y,N,2019-2020,15-1199


---------------------------
---------------------------

In [34]:
df_reduced["WAGE_UNIT_OF_PAY"].value_counts(dropna=False)

,count
WAGE_UNIT_OF_PAY,
Year,3344992
Hour,212172
Month,4519
Week,1739
Bi-Weekly,1274
NaN,2


In [35]:
# 2. Normalizar la unidad de pago
df_reduced["wage_unit_clean"] = (
    df_reduced["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 3. Diccionario de conversión a salario anual
unit_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080

}
# 4. Crear multiplicador
df_reduced["wage_unit_multiplier"] = df_reduced["wage_unit_clean"].map(unit_multiplier)

# 5. Convertir salario base a numérico
df_reduced["WAGE_UNIT_OF_PAY"] = pd.to_numeric(
    df_reduced["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# 6. Crear variable objetivo anualizada
df_reduced["offered_anual_avg_wage"] = (
    df_reduced["WAGE_RATE_OF_PAY_FROM"] * df_reduced["wage_unit_multiplier"]
)

df_reduced = df_reduced.drop(
    columns=["WAGE_RATE_OF_PAY_FROM", "WAGE_UNIT_OF_PAY", "wage_unit_clean", "wage_unit_multiplier"],
    errors="ignore"
)

In [36]:
display(df_reduced["offered_anual_avg_wage"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).apply('{:.2f}'.format))


,offered_anual_avg_wage
count,3564696.00
mean,279716.82
std,6803787.90
min,11.80
1%,46508.80
5%,60000.00
10%,69553.00
25%,85000.00
50%,106950.00
75%,140000.00


In [37]:
# Filtra y muestra las filas en el rango deseado
df_limit_wage = df_reduced[df_reduced["offered_anual_avg_wage"].between(15000, 300000)].copy()
df_limit_wage = df_limit_wage.reset_index(drop=True)

In [38]:
df_limit_wage["offered_anual_avg_wage"].describe()


,offered_anual_avg_wage
count,3.533032e+06
mean,1.154303e+05
std,4.323961e+04
min,1.508000e+04
25%,8.480200e+04
50%,1.060800e+05
75%,1.394640e+05
max,3.000000e+05


-----------------------------
-----------------------------

## Aplicaremos la misma lógica a la columna "PREVAILING_WAGE"

In [39]:
# 1. Normalizar la unidad de pago del Prevailing Wage
df_limit_wage["pw_unit_clean"] = (
    df_limit_wage["PW_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 2. Diccionario de conversión a salario anual
unit_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# 3. Crear multiplicador (Usamos fillna(1) para que si no hay unidad, asuma que ya es anual)
df_limit_wage["pw_unit_multiplier"] = df_limit_wage["pw_unit_clean"].map(unit_multiplier).fillna(1)

# 4. Convertir salario legal (Prevailing Wage) a numérico de forma segura
df_limit_wage["PREVAILING_WAGE"] = pd.to_numeric(
    df_limit_wage["PREVAILING_WAGE"],
    errors="coerce"
)

# 5. Sobrescribir la variable para que quede anualizada
df_limit_wage["PREVAILING_WAGE"] = (
    df_limit_wage["PREVAILING_WAGE"] * df_limit_wage["pw_unit_multiplier"]
)

# 6. Limpiar la "basura" temporal para no ensuciar el modelo
df_limit_wage = df_limit_wage.drop(
    columns=["pw_unit_clean", "pw_unit_multiplier", "PW_UNIT_OF_PAY"],
    errors="ignore"
)

# 7.
df_limit_wage = df_limit_wage[df_limit_wage["PREVAILING_WAGE"].between(15000, 300000)].copy()
df_limit_wage = df_limit_wage.reset_index(drop=True)

----------------------------
----------------------------

## Analizamos la columna "NEW_EMPLOYMENT"

In [40]:
df_limit_wage["NEW_EMPLOYMENT"].value_counts()

,count
NEW_EMPLOYMENT,
0,2396146
1,1042761
5,20999
2,17456
10,14915
...,...
37,1
1098,1
223,1


In [41]:
df_limit_wage[df_limit_wage["NEW_EMPLOYMENT"]>0]["NEW_EMPLOYMENT"].value_counts()

,count
NEW_EMPLOYMENT,
1,1042761
5,20999
2,17456
10,14915
4,9023
...,...
37,1
1098,1
223,1


## En base al análsis previos, tomamos la decisión de trabajar cuando NEW_EMPLOYMENT > 0 y CASE_STATUS == "Certified", para asegurar que la solicitud sea de nuevos empleados y solicitudes certificadas.

In [42]:
df_filtrado = df_limit_wage[
    (df_limit_wage["CASE_STATUS"] == "Certified") &
    (df_limit_wage["NEW_EMPLOYMENT"] > 0) &
    (df_limit_wage["TOTAL_WORKER_POSITIONS"] == df_limit_wage["NEW_EMPLOYMENT"])
].copy()

print("Dimensiones del dataset filtrado:")
print(df_filtrado.shape)


Dimensiones del dataset filtrado:
(982863, 31)


#### Añadimos la siguiente lógica donde "TOTAL_WORKER_POSITIONS" == "NEW_EMPLOYMENT" así garantizamos trabajar exclusivamente con los procesos puramente de nuevos empleados.

In [43]:
df_filtrado["NEW_EMPLOYMENT"].value_counts().sort_index()

,count
NEW_EMPLOYMENT,
1,935262
2,4889
3,2142
4,1213
5,7683
...,...
200,1
250,14
275,3


In [44]:
print("Validación:")
print("Casos Certificados:", (df_filtrado["CASE_STATUS"] == "Certified").all())
print("Todos con NEW_EMPLOYMENT > 0:", (df_filtrado["NEW_EMPLOYMENT"] > 0).all())
print(
    "TOTAL_WORKER_POSITIONS == NEW_EMPLOYMENT:",
    (df_filtrado["TOTAL_WORKER_POSITIONS"] == df_filtrado["NEW_EMPLOYMENT"]).all()
)

Validación:
Casos Certificados: True
Todos con NEW_EMPLOYMENT > 0: True
TOTAL_WORKER_POSITIONS == NEW_EMPLOYMENT: True


Se filtró el dataset para trabajar únicamente con solicitudes certificadas correspondientes exclusivamente a nuevos empleados. Esta decisión permite enfocar el proyecto en un escenario empresarial claro: estimar el salario anual mínimo ofrecido en procesos de nueva contratación de talento extranjero.

Para ello, se conservaron solo los registros donde CASE_STATUS es Certified, NEW_EMPLOYMENT es mayor que cero y donde TOTAL_WORKER_POSITIONS coincide con NEW_EMPLOYMENT. Esto asegura que la solicitud no mezcla otros tipos de peticiones, como continuidad laboral(CONTINUED_EMPLOYMENT), cambio de empleador (CHANGE_PREVIOUS_EMPLOYMENT), empleo concurrente (NEW_CONCURRENT_EMPLOYMENT) o peticiones enmendadas (AMENDED_PETITION).

### Validamos columnas con fuerte relación con "NEW_EMPLOYMENT"

In [45]:
cols_reviews = [
    "CONTINUED_EMPLOYMENT",
    "CHANGE_PREVIOUS_EMPLOYMENT",
    "NEW_CONCURRENT_EMPLOYMENT",
    "CHANGE_EMPLOYER",
    "AMENDED_PETITION"
]

for col in cols_reviews:
    print(f"\n{col}")
    print(df_filtrado[col].value_counts(dropna=False).head())


CONTINUED_EMPLOYMENT
CONTINUED_EMPLOYMENT
0     982272
1        542
2         26
5          9
10         4
Name: count, dtype: int64

CHANGE_PREVIOUS_EMPLOYMENT
CHANGE_PREVIOUS_EMPLOYMENT
0    982302
1       526
2        19
4         4
5         3
Name: count, dtype: int64

NEW_CONCURRENT_EMPLOYMENT
NEW_CONCURRENT_EMPLOYMENT
0    982423
1       382
2        30
4         9
3         7
Name: count, dtype: int64

CHANGE_EMPLOYER
CHANGE_EMPLOYER
0    980521
1      2273
2        36
5        12
6         8
Name: count, dtype: int64

AMENDED_PETITION
AMENDED_PETITION
0      982236
1         582
2          24
250         5
4           5
Name: count, dtype: int64


Todavía existen algunos registros donde las otras columnas de petición tienen valores distintos de 0.
Filtraremos los datos para que quede enteramente en valores 0 y eliminaremos esas columnas ya que no nos entregan información importante.

In [46]:
df_filtrado_final = df_filtrado[
    (df_filtrado["CONTINUED_EMPLOYMENT"] == 0) &
    (df_filtrado["CHANGE_PREVIOUS_EMPLOYMENT"] == 0) &
    (df_filtrado["NEW_CONCURRENT_EMPLOYMENT"] == 0) &
    (df_filtrado["CHANGE_EMPLOYER"] == 0) &
    (df_filtrado["AMENDED_PETITION"] == 0)
].copy()


In [47]:
df_filtrado_final.head()

,CASE_STATUS,RECEIVED_DATE,VISA_CLASS,JOB_TITLE,SOC_TITLE,FULL_TIME_POSITION,BEGIN_DATE,END_DATE,TOTAL_WORKER_POSITIONS,NEW_EMPLOYMENT,...,WORKSITE_COUNTY,WORKSITE_STATE,PREVAILING_WAGE,PW_WAGE_LEVEL,TOTAL_WORKSITE_LOCATIONS,H_1B_DEPENDENT,WILLFUL_VIOLATOR,pw_oes_period_group,soc_code_grouped,offered_anual_avg_wage
0,Certified,2019-09-25,H-1B,"APPLICATION ENGINEER, OMS [15-1199.02]","COMPUTER OCCUPATIONS, ALL OTHER",Y,2019-10-07,2022-10-07,1,1,...,Summit,OH,95118.0,IV,NaN,N,N,2018-2019,15-1199,100000.0
855,Certified,2019-09-25,H-1B,SOFTWARE ENGINEER,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,2019-10-14,2022-10-13,1,1,...,ANNE ARUNDEL COUNTY,MD,81474.0,II,NaN,Y,N,2019-2020,15-1132,81500.0
856,Certified,2019-09-25,H-1B,SOFTWARE DEVELOPER,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,2019-09-25,2022-09-24,1,1,...,Middlesex,MA,91520.0,II,NaN,Y,N,2019-2020,15-1132,91873.6
857,Certified,2019-09-25,H-1B,DESIGN ENGINEER,COMMERCIAL AND INDUSTRIAL DESIGNERS,Y,2019-10-30,2022-10-29,1,1,...,Los Angeles,CA,60528.0,II,NaN,Y,N,2019-2020,OTHER,70000.0
858,Certified,2019-09-25,H-1B,PROGRAMMER/ DEVELOPER 2,COMPUTER PROGRAMMERS,Y,2019-09-25,2022-09-24,1,1,...,MCLEAN COUNTY,IL,65312.0,II,NaN,Y,N,2019-2020,15-1131,65400.0


In [48]:
df_filtrado_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 979113 entries, 0 to 3531070
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   CASE_STATUS                 979113 non-null  object 
 1   RECEIVED_DATE               979113 non-null  object 
 2   VISA_CLASS                  979113 non-null  object 
 3   JOB_TITLE                   979113 non-null  object 
 4   SOC_TITLE                   979113 non-null  object 
 5   FULL_TIME_POSITION          979113 non-null  object 
 6   BEGIN_DATE                  979113 non-null  object 
 7   END_DATE                    979113 non-null  object 
 8   TOTAL_WORKER_POSITIONS      979113 non-null  int64  
 9   NEW_EMPLOYMENT              979113 non-null  int64  
 10  CONTINUED_EMPLOYMENT        979113 non-null  int64  
 11  CHANGE_PREVIOUS_EMPLOYMENT  979113 non-null  int64  
 12  NEW_CONCURRENT_EMPLOYMENT   979113 non-null  int64  
 13  CHANGE_EMPLOYER   

In [49]:
df_filtrado_final['NEW_EMPLOYMENT'].value_counts()


,count
NEW_EMPLOYMENT,
1,931692
10,10523
5,7667
2,4812
25,4414
...,...
42,1
64,1
33,1


In [50]:
df_filtrado_final = df_filtrado_final.drop(
    columns=[
        "CASE_STATUS",
        "JOB_TITLE",
        "CONTINUED_EMPLOYMENT",
        "CHANGE_PREVIOUS_EMPLOYMENT",
        "NEW_CONCURRENT_EMPLOYMENT",
        "CHANGE_EMPLOYER",
        "AMENDED_PETITION",
        "EMPLOYER_NAME",
        "SOC_TITLE",
        "TOTAL_WORKER_POSITIONS"
    ],
    errors="ignore"
)


In [51]:
col_cat = df_filtrado_final.select_dtypes(include=['object', 'string']).columns
df_filtrado_final[col_cat] = df_filtrado_final[col_cat].astype('category')

In [52]:
df_filtrado_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 979113 entries, 0 to 3531070
Data columns (total 21 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   RECEIVED_DATE             979113 non-null  category
 1   VISA_CLASS                979113 non-null  category
 2   FULL_TIME_POSITION        979113 non-null  category
 3   BEGIN_DATE                979113 non-null  category
 4   END_DATE                  979113 non-null  category
 5   NEW_EMPLOYMENT            979113 non-null  int64   
 6   EMPLOYER_STATE            978966 non-null  category
 7   NAICS_CODE                979113 non-null  int64   
 8   WORKSITE_WORKERS          978199 non-null  float64 
 9   SECONDARY_ENTITY          979113 non-null  category
 10  WORKSITE_CITY             979100 non-null  category
 11  WORKSITE_COUNTY           979021 non-null  category
 12  WORKSITE_STATE            979113 non-null  category
 13  PREVAILING_WAGE           979113 

---------------------------
---------------------------

In [53]:
df_final = df_filtrado_final.copy()

# 2. Forzar que la variable objetivo sea un número.
df_final['offered_anual_avg_wage'] = pd.to_numeric(df_final['offered_anual_avg_wage'], errors='coerce')

# 3. np.isfinite() detecta y conserva solo números reales (elimina NaN e Inf)
df_final = df_final[np.isfinite(df_final['offered_anual_avg_wage'])]

df_final = df_final.reset_index(drop=True)

In [54]:
df_final['offered_anual_avg_wage'].describe()


,offered_anual_avg_wage
count,979113.000000
mean,100366.635484
std,39857.473032
min,15080.000000
25%,74277.000000
50%,91104.000000
75%,117500.000000
max,300000.000000


# Paso 6: Construye el modelo y optimízalo

Selección y entrenamiento del algoritmo que mejor se ajuste a la naturaleza del problema. Se incluye un proceso de optimización de hiperparámetros para ajustar el modelo y alcanzar el máximo rendimiento posible en sus métricas de evaluación.

In [55]:
# Separamos las variables independientes y la variable objetivo en "x" y "y".

X = df_final.drop(columns = ['offered_anual_avg_wage'])
y = df_final['offered_anual_avg_wage']

In [56]:
# Usamos train_test_split para separar la data de testeo y de entrenamiento.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [57]:
model = xgb.XGBRegressor(
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    random_state=42
)

model.fit(X_train, y_train)

# 3. Predicción
y_pred = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning:

[02:25:41] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.




In [58]:
# Evaluar el desempeño básico
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE (Error Cuadrático Medio): {mse:,.2f}")
print(f"RMSE (Raíz del Error Cuadrático): ${rmse:,.2f}")
print(f"R2 (Coeficiente de Determinación): {r2:.4f}")

MSE (Error Cuadrático Medio): 241,705,210.61
RMSE (Raíz del Error Cuadrático): $15,546.87
R2 (Coeficiente de Determinación): 0.8496


### Generamos un dataset de prueba de 100k datos para evaluar el modelo optimizado

In [59]:
df_100, _ = train_test_split(df_final, train_size=100000, random_state=42, shuffle=True)
df_100 = df_100.reset_index(drop=True)
df_100.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 21 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   RECEIVED_DATE             100000 non-null  category
 1   VISA_CLASS                100000 non-null  category
 2   FULL_TIME_POSITION        100000 non-null  category
 3   BEGIN_DATE                100000 non-null  category
 4   END_DATE                  100000 non-null  category
 5   NEW_EMPLOYMENT            100000 non-null  int64   
 6   EMPLOYER_STATE            99985 non-null   category
 7   NAICS_CODE                100000 non-null  int64   
 8   WORKSITE_WORKERS          99904 non-null   float64 
 9   SECONDARY_ENTITY          100000 non-null  category
 10  WORKSITE_CITY             99998 non-null   category
 11  WORKSITE_COUNTY           99993 non-null   category
 12  WORKSITE_STATE            100000 non-null  category
 13  PREVAILING_WAGE           1000

In [60]:
# Separamos las variables independientes y la variable objetivo en "x" y "y".

X = df_100.drop(columns = ['offered_anual_avg_wage'])
y = df_100['offered_anual_avg_wage']

In [61]:
# Usamos train_test_split para separar la data de testeo y de entrenamiento.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## Generamos modelo optimizado utilizando Optuna

In [62]:
def objective(trial):
    param = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda",
        "enable_categorical": True,
        "random_state": 42,

        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),

        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 10.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),

        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 32),
    }

    model = xgb.XGBRegressor(
        **param,
        early_stopping_rounds=50
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    preds = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, preds))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(f"Mejor RMSE: {study.best_value:,.2f}")
print(f"Mejores parámetros: {study.best_params}")

[I 2026-05-12 02:25:41,578] A new study created in memory with name: no-name-f2bdc457-c18c-44d8-9c8b-0e0c70a0e076
[I 2026-05-12 02:25:50,674] Trial 0 finished with value: 17799.343685464893 and parameters: {'n_estimators': 585, 'max_depth': 5, 'learning_rate': 0.01939747872545481, 'subsample': 0.8978719810537988, 'colsample_bytree': 0.7276703612259542, 'min_child_weight': 1.0174338483737602, 'gamma': 8.288438679828268, 'reg_alpha': 1.532029623909635e-06, 'reg_lambda': 5.037748021511376, 'max_cat_to_onehot': 25}. Best is trial 0 with value: 17799.343685464893.
[I 2026-05-12 02:26:14,954] Trial 1 finished with value: 17499.814648605057 and parameters: {'n_estimators': 903, 'max_depth': 7, 'learning_rate': 0.03553343495187531, 'subsample': 0.6846196468007335, 'colsample_bytree': 0.7556819643255526, 'min_child_weight': 3.6234451395762846, 'gamma': 4.2510505720398415, 'reg_alpha': 0.9006945750376352, 'reg_lambda': 8.4364749688515, 'max_cat_to_onehot': 4}. Best is trial 1 with value: 17499.8

Mejor RMSE: 16,709.24
Mejores parámetros: {'n_estimators': 1038, 'max_depth': 9, 'learning_rate': 0.0575995675697574, 'subsample': 0.9992285209034455, 'colsample_bytree': 0.830766272283785, 'min_child_weight': 9.514343558520656, 'gamma': 0.25897606467589185, 'reg_alpha': 6.295494556095613e-06, 'reg_lambda': 0.6267020112002192, 'max_cat_to_onehot': 15}


## Evaluamos dataset final con el estudio generado mediante Optuna

In [63]:
X = df_final.drop(columns = ['offered_anual_avg_wage'])
y = df_final['offered_anual_avg_wage']

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [65]:
final_params = study.best_params.copy()

# 2. Añadimos los parámetros técnicos que no se optimizan (GPU, categorías, etc.)
final_params.update({
    "tree_method": "hist",
    "device": "cuda",
    "enable_categorical": True,
    "random_state": 42,
    "early_stopping_rounds": 50
})

# 3. Instanciamos el modelo usando el desempaquetado de diccionario (**)
final_model = xgb.XGBRegressor(**final_params)

# 4. Entrenamos el modelo
# TIP: Si quieres bajar más el RMSE, intenta usar aquí el dataset completo
# en lugar de la muestra de 100k, ya que tienes los parámetros ideales.
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

# 5. Evaluación final
final_preds = final_model.predict(X_test)
rmse_final = np.sqrt(mean_squared_error(y_test, final_preds))

print(f"Métricas del Modelo Final:")
print(f"RMSE: {rmse_final}")

[0]	validation_0-rmse:38952.60980
[100]	validation_0-rmse:15412.79199
[200]	validation_0-rmse:14907.89737
[300]	validation_0-rmse:14624.01570
[400]	validation_0-rmse:14418.02057
[500]	validation_0-rmse:14269.63564
[600]	validation_0-rmse:14158.72567
[700]	validation_0-rmse:14059.02264
[800]	validation_0-rmse:13982.15106
[900]	validation_0-rmse:13910.84739
[1000]	validation_0-rmse:13847.33467
[1037]	validation_0-rmse:13826.06963
Métricas del Modelo Final:
RMSE: 13826.25484310764


---------------------------

## Guardamos Modelo

In [66]:
import pickle

# Ruta completa donde se guardará el modelo
model_path = '/content/drive/MyDrive/Colab Notebooks/PROYECTO 4GEEKS/optuna_20trials_model.pkl'

# Guardar modelo con pickle
with open(model_path, 'wb') as file:
    pickle.dump(final_model, file)

print(f'✅ Modelo guardado correctamente en: {model_path}')

✅ Modelo guardado correctamente en: /content/drive/MyDrive/Colab Notebooks/PROYECTO 4GEEKS/optuna_20trials_model.pkl


## Paso 7: Despliega el modelo

Fase final donde se crea una interfaz de usuario, habitualmente una aplicación web (usando Flask o Streamlit), y se implementa en una plataforma en la nube como Render o Heroku. Esto permite que el modelo sea accesible para usuarios finales o clientes potenciales.